In [ ]:
*Title*: 2_Allele_Frequency_Differences_and_Outliers
*Goal*: Generate files for either outlier SNPs (based on FST) and/or neutral SNPs. Also, generate allele frequency difference files for 2016 and 2017.
Based on original script: PCA_outliers

# 2.1 Generate allele frequency differences  ----------------------
Showing for 2016 only, but 2017 same

``` bash
module load StdEnv/2020
perl /home/mary2018/.local/bin/popoolation2_1201/snp-frequency-diff.pl --input /home/mary2018/projects/rrg-barrett/mary2018/MultiYear/analysis/masked/REDO/1.2016masked-autosome-chrom-reorder.sync --output-prefix 5.2016masked-autosome-allele-freq-diff  --min-count 2 --min-coverage 5 --max-coverage 100 
``` 
# 2.2 Restructure the file  ----------------------
The  output file lists SNPs that violate the filtering parameters as 0 (i.e. coverage > 100, the output file recomputes it to be 0/101) instead of removing the line altogether! So removing lines that violate filtering parameters. Also rearranging the file to match the structure of other files.

``` R
library("tidyverse")

#2016 file generation:
allele_freq_data2016 <- read.table("5.2016masked-autosome-allele-freq-diff_rc", header = FALSE) %>%
  mutate(Chromosome_Position = paste(V1, V2, sep = "_")) %>%   # Combine 'chr' and 'pos' columns into 'Chromosome_Position'
  select(Chromosome_Position, V10:V21) #I want to select the major allele columns 

colnames(allele_freq_data2016)<- c("Chromosome_Position", "Younger_Spring-2016","Younger_Fall-2016","OldDairy_Spring-2016","OldDairy_Fall-2016","Scott_Spring-2016","Scott_Fall-2016",
"Lombardi_Spring-2016","Lombardi_Fall-2016","Laguna_Spring-2016","Laguna_Fall-2016","Waddell_Spring-2016","Waddell_Fall-2016")

write.table(allele_freq_data2016,"allele_freq_data_2016.tsv", sep = "\t",row.names=FALSE, quote =FALSE)
``` 
# 2.3 Filtering ----------------------
#Generate 2016 and 2017 files that are either neutral (no outliers of that year) OR outliers
#outlier file will show how similar/different selection is across populations
#neutral will show population structure
#Filtering both files to only include relevant SNPs:

# Keep ALL SNPs present in both years (whether neutral or outliers):
```bash
  #First I have to get the column names, but I dont want the first column (contains population names):
  #The output file is reorganized so that the columns are now rows in 1 column called chromosome_position:

  #Now filter each file to only include the Snps:
  wc -l allele_freq_data_2016.tsv #9020067 Snps (representing alleles from 2016 populations assessed for major alleles)
 awk 'FILENAME=="all_snps.tsv" {a[$1]; next} $1 in a' all_snps.tsv allele_freq_data_2016.tsv > allele_freq_data_2016_neutral_and_outliers.tsv 
 
  wc -l allele_freq_data_2016_neutral_and_outliers.tsv 
  
  wc -l allele_freq_data_2017.tsv #=24553592
  awk 'FILENAME=="all_snps.tsv" {a[$1]; next} $1 in a' all_snps.tsv allele_freq_data_2017.tsv > allele_freq_data_2017_neutral_and_outliers.tsv 
  wc -l allele_freq_data_2017_neutral_and_outliers.tsv 

#Note that the wc-l are not exactly the same. Since all_snps was based off the poolfstat fst outliers and the all_freq is from the popoolation allele_freq, it is possible that both programs have slightly different results despite the same filtering parameters.

# Outliers Only, Neutral Only
#2016 outliers
  #filtering to only include the outliers
  awk 'FILENAME=="2016_outliers.tsv" {a[$1]; next} $1 in a' 2016_outliers.tsv allele_freq_data_2016_neutral_and_outliers.tsv > allele_freq_data_2016_outliers.tsv
  wc -l allele_freq_data_2016_outliers.tsv # 1572903 Snps

  
#2016 neutral
  #filtering to NOT include the outliers
(head -n 1 allele_freq_data_2016_neutral_and_outliers.tsv && awk 'FILENAME=="2016_outliers.tsv" {a[$1]; next} NR>1 && !($1 in a)' 2016_outliers.tsv allele_freq_data_2016_neutral_and_outliers.tsv) > allele_freq_data_2016_neutral.tsv


  #awk 'FILENAME=="2016_outliers.tsv" {a[$1]; next} !($1 in a)' 2016_outliers.tsv allele_freq_data_2016_neutral_and_outliers.tsv > allele_freq_data_2016_neutral.tsv
  wc -l allele_freq_data_2016_neutral.tsv 
  ```

# 2.4 Cleaning the file  ---------------------- 
*Goal:* Clean all files made above using a loop. Separate versions for both years, as they both have different filtering restrictions. 

# 2017 version:
  ``` R
library(data.table)
check_greater_parameters <- function(row) {
  results <- sapply(row, function(cell) {
    parts <- strsplit(as.character(cell), "/")[[1]]
    if(length(parts) < 2) return(FALSE)
    num <- as.numeric(parts[1])
    denom <- as.numeric(parts[2])
    denom < 5 || denom > 300
  })
  any(results)
}

file_list <- list.files(pattern = "allele_freq_data_2017.*\\.tsv")

#Loop through each file
for (file in file_list) {
  #Read the data with column and row names
  allele_freq_data <- read.table(file, header = TRUE, sep = "\t", stringsAsFactors = FALSE, row.names = 1)

  #Check for NA before alterations
  print(paste("Before alteration for file:", file))
  base_name <- sub("\\.tsv$", "", basename(file))
  print("number of NA")
  print(sum(is.na(allele_freq_data)))
  print("number of SNPS:")
  print(nrow(allele_freq_data))

  #Remove rows that didn't pass max coverage limit
  allele_freq_data <- allele_freq_data[!apply(allele_freq_data, 1, check_greater_parameters), ]

  print("after coverage filtering")
  print("number of NA")
  print(sum(is.na(allele_freq_data)))
  print("number of SNPS:")
   print(nrow(allele_freq_data))

  #Fraction conversion
row_names <- row.names(allele_freq_data)

allele_freq_data <- as.data.frame(lapply(allele_freq_data, function(x) {
  sapply(strsplit(as.character(x), "/"), function(y) as.numeric(y[1]) / as.numeric(y[2]))
}))

#Reassign the row names
row.names(allele_freq_data) <- row_names
  print("after convert from fraction")
  print("number of NA")
  print(sum(is.na(allele_freq_data)))
 print("number of SNPS:")
   print(nrow(allele_freq_data))


  #Transpose the data to have samples as rows and SNPs as columns
  allele_freq_data <- as.data.frame(t(allele_freq_data))
  print("after transposition:")
  #The the snps are now the columns
  print(allele_freq_data[1:2, 1:2])
  print(class(allele_freq_data))

#To look at number of constants removed:
print("number of snps before constants removed")
print(ncol(allele_freq_data))

  #Identify and remove constant/zero-variance columns
  constant_cols <- apply(allele_freq_data, 2, function(x) length(unique(x)) == 1)
  constant_cols_indices <- which(constant_cols)
  allele_freq_data <- allele_freq_data[, !constant_cols]
  constant_cols_data <- data.frame(Column_Index = constant_cols_indices)
  write.table(as.data.frame(constant_cols_data), paste0(base_name, "_constant_cols.tsv"), sep = "\t", row.names = TRUE, col.names = TRUE, quote = FALSE)
  print("number of snps after constants removed")
print(ncol(allele_freq_data))
  print(allele_freq_data[1:2, 1:2])
  print(class(allele_freq_data))
  print(sum(is.na(allele_freq_data)))
allele_freq_data$population <- row.names(allele_freq_data)
allele_freq_data <- allele_freq_data[, c("population", setdiff(names(allele_freq_data), "population"))] #doing this so i can have the pop names saved

fwrite(allele_freq_data, paste0(base_name, "_no_constants.tsv"), sep = "\t", quote = FALSE)
print("done")
}
 ```

# 2016 version:
``` R
library(data.table)

check_greater_parameters <- function(row) {
  results <- sapply(row, function(cell) {
    parts <- strsplit(as.character(cell), "/")[[1]]
    if(length(parts) < 2) return(FALSE)
    num <- as.numeric(parts[1])
    denom <- as.numeric(parts[2])
    denom < 5 || denom > 100
  })
  any(results)
}

file_list <- list.files(pattern = "allele_freq_data_2016.*\\.tsv")

# Loop through each file
for (file in file_list) {
  # Read the data with column and row names
  allele_freq_data <- read.table(file, header = TRUE, sep = "\t", stringsAsFactors = FALSE, row.names = 1)

  # Check for NA before alterations
  print(paste("Before alteration for file:", file))
  base_name <- sub("\\.tsv$", "", basename(file))
  print("number of NA")
  print(sum(is.na(allele_freq_data)))
  print("number of SNPS:")
  print(nrow(allele_freq_data))

  # Remove rows that didn't pass max coverage limit
  allele_freq_data <- allele_freq_data[!apply(allele_freq_data, 1, check_greater_parameters), ]

  print("after coverage filtering")
  print("number of NA")
  print(sum(is.na(allele_freq_data)))
  print("number of SNPS:")
   print(nrow(allele_freq_data))

  # Fraction conversion
row_names <- row.names(allele_freq_data)

allele_freq_data <- as.data.frame(lapply(allele_freq_data, function(x) {
  sapply(strsplit(as.character(x), "/"), function(y) as.numeric(y[1]) / as.numeric(y[2]))
}))

# Reassign the row names
row.names(allele_freq_data) <- row_names
  print("after convert from fraction")
  print("number of NA")
  print(sum(is.na(allele_freq_data)))
 print("number of SNPS:")
   print(nrow(allele_freq_data))

  print("after transposition:")

  # Transpose the data to have samples as rows and SNPs as columns
  allele_freq_data <- as.data.frame(t(allele_freq_data))

  #The the snps are now the columns
  print(allele_freq_data[1:2, 1:2])
  print(class(allele_freq_data))

# To look at number of constants removed:
print("number of snps before constants removed")
print(ncol(allele_freq_data))

  # Identify and remove constant/zero-variance columns
  constant_cols <- apply(allele_freq_data, 2, function(x) length(unique(x)) == 1)
  constant_cols_indices <- which(constant_cols)
  allele_freq_data <- allele_freq_data[, !constant_cols]
  constant_cols_data <- data.frame(Column_Index = constant_cols_indices)
  write.table(as.data.frame(constant_cols_data), paste0(base_name, "_constant_cols.tsv"), sep = "\t", row.names = TRUE, col.names = TRUE, quote = FALSE)
  print("number of snps after constants removed")
print(ncol(allele_freq_data))
  print(allele_freq_data[1:2, 1:2])
  print(class(allele_freq_data))
  print(sum(is.na(allele_freq_data)))
allele_freq_data$population <- row.names(allele_freq_data)
allele_freq_data <- allele_freq_data[, c("population", setdiff(names(allele_freq_data), "population"))] #doing this so i can have the pop names saved

fwrite(allele_freq_data, paste0(base_name, "_no_constants.tsv"), sep = "\t", quote = FALSE)
print("done")
}
```
#Redid Feb. 16, 2025.

# Graphing  ----------------------
#updating graphing using a package instead of manually
#script inspo: https://www.sthda.com/english/articles/31-principal-component-methods-in-r-practical-guide/118-principal-component-analysis-in-r-prcomp-vs-princomp/
#not looping because too slow.
#submitting each population individually (see scripts: allele_freq_data_2016_*_no_constants_pca.R, wheere * refers to the input file specifics)
#pwd: /home/mary2018/projects/rrg-barrett/mary2018/MultiYear/analysis/masked/REDO/poolfstat-fst-version-ii/pca_processing_generation
#Please note that version 1 used keys and I think that messed up the row identites. Made version 2 without the keys!

#Did the same thing for allele_freq_data_2016_outliers_no_constants.tsv

cut -f1 allele_freq_data_2017_outliers_no_constants.tsv > allele_freq_data_2017_outliers_no_constants_populations2.tsv #making a file with population names

#in R:
library(ggplot2)
library(factoextra)
library(tidyverse)
library(dplyr)
library(data.table)

## 2017 outliers:

#Read the data
file<- "allele_freq_data_2017_outliers_no_constants.tsv"
allele_freq_data <- fread(file, header = TRUE, sep = "\t")

print("data loaded")

#Perform PCA
pca_result <- prcomp(allele_freq_data[, -1], center = TRUE, scale = TRUE) #run pca on everything except text (population names in first column)

#Save pca results
saveRDS(pca_result, file = paste0(sub(".tsv", "", file), "_pca_result.rds"))
write.table(pca_result$sdev, file = paste0(sub(".tsv", "", file), "_pca_result_sd.tsv"))
write.table(pca_result$rotation, file = paste0(sub(".tsv", "", file), "_pca_result_rotation.tsv"))
write.table(pca_result$x, file = paste0(sub(".tsv", "", file), "_pca_result_scores.tsv"))

#Interactive JOB! (manually changed each file and re-ran)
library(ggplot2)
library(factoextra)
library(tidyverse)
library(dplyr)
library(data.table)

pca_result<-readRDS("allele_freq_data_2017_outliers_no_constants_pca_result.rds")
file<- "allele_freq_data_2017_outliers_no_constants_populations.tsv"

allele_freq_data <- fread(file, header = TRUE, sep = "\t") 
rownames(pca_result$x) <- allele_freq_data$population

#Process the data
allele_freq_data[, estuary := sapply(strsplit(population, "_"), `[`, 1)]
allele_freq_data[, season := sapply(strsplit(population, "_|-"), `[`, 2)]
allele_freq_data[, season := as.factor(season)]
allele_freq_data[, estuary := as.factor(estuary)]

#Scree plot
scree_plot <- fviz_eig(pca_result)
ggsave(filename = paste0(sub(".tsv", "", file), "_scree_plot2.png"), plot = scree_plot, width = 8, height = 6, dpi = 300)

#Populations only (coloured by estuary, shape by year)
pc_scores <- as.data.table(pca_result$x[, 1:3])
pc_scores[, Population := allele_freq_data$population]

pc_scores[, estuary := sub("_.*", "", Population)]
pc_scores[, SampleDate := sub("^[^_]*_", "", Population)]


#Fix colour issue and add the Variance Explained on each PC:
explained_variance <- pca_result$sdev^2 / sum(pca_result$sdev^2) * 100
pc1_var <- round(explained_variance[1], 2)
pc2_var <- round(explained_variance[2], 2)


#Version 2:
#March 6, adding horizontal lines along the axes (Andrew feedback) AND arrows from Spring to Fall!!
#July 2, 2025- Reorganizing labels in legend because S and F mislabelled:

pc_scores <-pc_scores %>%
mutate(Season=sub("\\..*", "", SampleDate))

pca_colored_plot <- ggplot(pc_scores, aes(x = PC1, y = PC2, color = estuary, shape = Season)) +
  geom_point(size = 3, stroke = 1.2) +
  geom_hline(yintercept = 0, linetype = "solid", color = "gray") +  # Add solid horizontal line at y = 0
  geom_vline(xintercept = 0, linetype = "solid", color = "gray") +  # Add solid vertical line at x = 0
  guides(color = guide_legend(title = "Estuary"), shape = guide_legend(title = "Sample Date")) +
  scale_color_manual(values = c("Younger" = "#2E2585", "Scott" = "#34B6FF", "Lombardi" = "#EADA02", "OldDairy" = "#D55E00", "Laguna" = "#287762", "Waddell" = "#A000E0")) +
  scale_shape_manual(values = c("Spring" = 1, "Fall" = 19)) + 
  labs(x = paste0("PC1 (", pc1_var, "%)"), y = paste0("PC2 (", pc2_var, "%)")) + 
  theme_minimal() +
  theme(
    axis.line = element_line(color = "black")  # Add axis lines
  )

  #For arrows between samples, gathering relevant PCS and arranging in Spring -> Fall order:
  spring_fall_pairs <- pc_scores %>%
  filter(Season %in% c("Spring", "Fall")) %>%
  arrange(estuary, factor(Season, levels = c("Spring", "Fall")))

#Create a data frame for the paths
paths <- spring_fall_pairs %>%
  group_by(estuary) %>%
  reframe(
    x = PC1,
    y = PC2,
    Season = Season,
    .groups = 'drop'
  )

#Add the paths to the existing plot
pca_colored_plot_arrows <- pca_colored_plot +
  geom_path(data = paths, aes(x = x, y = y, group = estuary),
            arrow = arrow(length = unit(0.2, "cm")), color = "black")

# Save the plot with arrows
ggsave(filename = paste0(sub(".tsv", "", file), "_pca_plot2_legend_edited_again.png"), plot = pca_colored_plot_arrows, width = 8, height = 6, dpi = 700)
ggsave(filename = paste0(sub(".tsv", "", file), "_pca_plot2_legend_edited_again.pdf"), plot = pca_colored_plot_arrows, width = 8, height = 6)

#REDONE MARCH 7, 2025. (to have arrows and axes)
#redone June 12, 2025 to have a better legend. (legend version has *_legend_edited* name )
#copied new output files to /home/mary2018/projects/rrg-barrett/mary2018/MultiYear_FINAL_VERSION/analysis/masked/REDO/pub_figures/
#Redune July 2, 2025 because legend was messed up (spring and Fall mislabbeled)


#not relevant to MS!!!
## Multiyear Analyses
#Multiyear PCA, for neutral and outlier SNPs!
#Script in: allele_freq_data_2016_2017_neutral_and_outliers_no_constants_pca.R

library(data.table)
file2016 <- "allele_freq_data_2016_neutral_and_outliers_no_constants.tsv"
file2017 <- "allele_freq_data_2017_neutral_and_outliers_no_constants.tsv"
data1 <- fread(file2016, header = TRUE, sep = "\t")
data2 <- fread(file2017, header = TRUE, sep = "\t")

#Check if column names are the same
if (all(names(data1) == names(data2))) { 
  merged_data <- rbind(data1, data2)
  write.table(merged_data, file = "allele_freq_data_2016_2017_neutral_and_outliers_no_constants.tsv", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
} else {
  #Find the columns that do not match
  unmatched_columns <- setdiff(union(names(data1), names(data2)), intersect(names(data1), names(data2)))
  
  #Write the unmatched columns to a file
  write.table(unmatched_columns, file = "unmatched_columns.txt", row.names = FALSE, col.names = FALSE, quote = FALSE)
  
  #Find the common columns
  common_columns <- intersect(names(data1), names(data2))
  
  #Subset the data to only include common columns
  data1_common <- data1[, ..common_columns]
  data2_common <- data2[, ..common_columns]
  
  #Merge the common columns
  merged_data <- rbind(data1_common, data2_common)
  write.table(merged_data, file = "allele_freq_data_2016_2017_common_columns.tsv", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
}


file<- "allele_freq_data_2016_2017_common_columns.tsv"
allele_freq_data <- fread(file, header = TRUE, sep = "\t")

#Perform PCA
pca_result <- prcomp(allele_freq_data[, -1], center = TRUE, scale = TRUE) #run pca on everything except text (population names in first column)

#Save pca results
saveRDS(pca_result, file = paste0(sub(".tsv", "", file), "_pca_result.rds"))
write.table(pca_result$sdev, file = paste0(sub(".tsv", "", file), "_pca_result_sd.tsv"))
write.table(pca_result$rotation, file = paste0(sub(".tsv", "", file), "_pca_result_rotation.tsv"))
write.table(pca_result$x, file = paste0(sub(".tsv", "", file), "_pca_result_scores.tsv"))

cut -f1 allele_freq_data_2016_2017_common_columns.tsv > allele_freq_data_2016_2017_common_column_populations.tsv

  # Graphing (interactive job)

library(ggplot2)
library(factoextra)
library(tidyverse)
library(dplyr)
library(data.table)
pca_result<-readRDS("allele_freq_data_2016_2017_common_columns_pca_result.rds")
file<- "allele_freq_data_2016_2017_common_column_populations.tsv"

allele_freq_data <- fread(file, header = TRUE, sep = "\t") 
rownames(pca_result$x) <- allele_freq_data$population



# Process the data
allele_freq_data[, estuary := sapply(strsplit(population, "_"), `[`, 1)]
allele_freq_data[, season := sapply(strsplit(population, "_|-"), `[`, 2)]
allele_freq_data[, season := as.factor(season)]
allele_freq_data[, estuary := as.factor(estuary)]

# Scree plot
scree_plot <- fviz_eig(pca_result)
ggsave(filename = paste0(sub(".tsv", "", file), "_scree_plot2.png"), plot = scree_plot, width = 8, height = 6, dpi = 300)

# Populations only (coloured by estuary, shape by year)
pc_scores <- as.data.table(pca_result$x[, 1:3])
pc_scores[, Population := allele_freq_data$population]

pc_scores[, estuary := sub("_.*", "", Population)]
pc_scores[, SampleDate := sub("^[^_]*_", "", Population)]


#add the Variance Explained on each PC:
explained_variance <- pca_result$sdev^2 / sum(pca_result$sdev^2) * 100
pc1_var <- round(explained_variance[1], 2)
pc2_var <- round(explained_variance[2], 2)

pca_colored_plot <- ggplot(pc_scores, aes(x = PC1, y = PC2, color = estuary, shape = SampleDate)) +
  geom_point(size = 3, stroke = 1.2) +
  guides(color = guide_legend(title = "Estuary"), shape = guide_legend(title = "Sample Date")) +
  scale_color_manual(values = c("Younger" = "#2E2585", "Scott" = "#34B6FF", "Lombardi" = "#EADA02", "OldDairy" = "#D55E00", "Laguna" = "#287762", "Waddell" = "#A000E0")) +
  scale_shape_manual(values = c("Spring.2016" = 1, "Fall.2016" = 19, "Spring.2017" = 0, "Fall.2017" = 15)) +
  labs(x = paste0("PC1 (", pc1_var, "%)"), y = paste0("PC2 (", pc2_var, "%)")) + 
  theme_minimal() 


ggsave(filename = paste0(sub(".tsv", "", file), "_pca_plot2.png"), plot = pca_colored_plot, width = 8, height = 6, dpi = 300)

#NOTES:
#The reason there are unmatched oclumns is because allele-freq-diff runs slightly differently than fst program. Since allel-freq was computed on each year separately, some SNPS are missing from a given file, even though they are present in the all-snps file (generated from fst). It isn't a problem.

#Redone Feb 16, 2025.
# Extra Graphing SCripts ----------------------
#Not redone
# Multi year analyses, but outliers only
#Outliers includes any SNP which is an outlier at least once (in any estuary in any year):
#I tried to run this before, but only did it on outliers present in BOTH years (i.e. outliers in both years), so redoing it here. To see old script, go to junk section.

#1. Need to get the list of outliers from FST. USe file generated in upset plot
#Grab the SNP names (the first row- i.e column names):
awk 'NR == 1 {for (i = 1; i <= NF; i++) print $i}' /home/mary2018/scratch/MultiYear/analysis/masked/REDO/poolfstat-fst-version-ii/upset/4.outliers_poolfstat_SNP_FST_95_2016_2017_outliers_only.tsv > SNP_outliers_2016_or_2017.tsv 

#2. Now filter the allele freq file to only contain sites which are outliers:
#Note that the input file is filtered based on neutral or outliers from all-snps (the file generated from fst of poollfstat). It is possible that the files won't have exactly the same numbers because allele freq was made with popoolation2 and fst with poolfstat
#allele freq was made with popoolation2 and fst with poolfstat

awk '
FILENAME == "SNP_outliers_2016_or_2017.tsv" {
# Store each value from all rows of file1 in an associative array
a[$1]
next
}
FILENAME == "allele_freq_data_2016_2017_common_columns.tsv" {
# Process the header row to determine which columns to print
if (FNR == 1) {
header = $1 "\t" # Always include the first column (population)
for (i = 2; i <= NF; i++) {
if ($i in a) {
cols_to_print[i] = 1
header = header $i "\t"
}
}
print header
} else {
# Print the entire row for the matching columns
row = $1 "\t" # Always include the first column (population)
for (i = 2; i <= NF; i++) {
if (i in cols_to_print) {
row = row $i "\t"
}
}
print row
}
}
' SNP_outliers_2016_or_2017.tsv allele_freq_data_2016_2017_common_columns.tsv > allele_freq_data_2016_2017_common_columns_outliers_only.tsv


#Checking, wc -l SNP_outliers_2016_or_2017.tsv = 2616718
#awk -F'\t' '{print NF; exit}' allele_freq_data_2016_2017_common_columns_outliers_only.tsv = 2616715
#Almost identical so probably fine 
#Counts that are outliers in both = 540, 399

#Generate the population list:
cut -f1 allele_freq_data_2016_2017_common_columns_outliers_only.tsv > allele_freq_data_2016_2017_common_columns_outliers_only_populations.tsv

#3. Do the PCA (R script)
library("data.table")
file<- "allele_freq_data_2016_2017_common_columns_outliers_only.tsv"
allele_freq_data <- fread(file, header = TRUE, sep = "\t")
allele_freq_data<- allele_freq_data[,-2616715] #for some reason, an extra column was added at the end (contains NA). Need to remove it!

#Save pca results
saveRDS(pca_result, file = paste0(sub(".tsv", "", file), "_pca_result.rds"))
#write.table(pca_result$sdev, file = paste0(sub(".tsv", "", file), "_pca_result_sd.tsv"))
#write.table(pca_result$rotation, file = paste0(sub(".tsv", "", file), "_pca_result_rotation.tsv"))
#write.table(pca_result$x, file = paste0(sub(".tsv", "", file), "_pca_result_scores.tsv"))

#4. Graph (interactive)

library(ggplot2)
library(factoextra)
library(tidyverse)
library(dplyr)
library(data.table)
pca_result<-readRDS("allele_freq_data_2016_2017_common_columns_outliers_only_pca_result.rds")
file<- "allele_freq_data_2016_2017_common_columns_outliers_only_populations.tsv"

allele_freq_data <- fread(file, header = TRUE, sep = "\t") 
rownames(pca_result$x) <- allele_freq_data$population

#Process the data
allele_freq_data[, estuary := sapply(strsplit(population, "_"), `[`, 1)]
allele_freq_data[, season := sapply(strsplit(population, "_|-"), `[`, 2)]
allele_freq_data[, season := as.factor(season)]
allele_freq_data[, estuary := as.factor(estuary)]

#Scree plot
scree_plot <- fviz_eig(pca_result)
ggsave(filename = paste0(sub(".tsv", "", file), "_scree_plot2.png"), plot = scree_plot, width = 8, height = 6, dpi = 300)

#Populations only (coloured by estuary, shape by year)
pc_scores <- as.data.table(pca_result$x[, 1:3])
pc_scores[, Population := allele_freq_data$population]

pc_scores[, estuary := sub("_.*", "", Population)]
pc_scores[, SampleDate := sub("^[^_]*_", "", Population)]


#add the Variance Explained on each PC:
explained_variance <- pca_result$sdev^2 / sum(pca_result$sdev^2) * 100
pc1_var <- round(explained_variance[1], 2)
pc2_var <- round(explained_variance[2], 2)

pca_colored_plot <- ggplot(pc_scores, aes(x = PC1, y = PC2, color = estuary, shape = SampleDate)) +
  geom_point(size = 3, stroke = 1.2) +
  guides(color = guide_legend(title = "Estuary"), shape = guide_legend(title = "Sample Date")) +
  scale_color_manual(values = c("Younger" = "#2E2585", "Scott" = "#34B6FF", "Lombardi" = "#EADA02", "OldDairy" = "#D55E00", "Laguna" = "#287762", "Waddell" = "#A000E0")) +
  scale_shape_manual(values = c("Spring.2016" = 1, "Fall.2016" = 19, "Spring.2017" = 0, "Fall.2017" = 15)) +
  labs(x = paste0("PC1 (", pc1_var, "%)"), y = paste0("PC2 (", pc2_var, "%)")) + 
  theme_minimal() 


ggsave(filename = paste0(sub(".tsv", "", file), "_pca_plot2.png"), plot = pca_colored_plot, width = 8, height = 6, dpi = 300)



########### JUNK?
#Mini version to check the colouring of graph:
#Ran this version (which is based on the original PCA method used on the popoolation stuff), to check if results would be the same. They are.
#MAde a mini file to practice:  head -n 14 allele_freq_data_2016_outliers_no_constants.tsv | cut -f1,2,3,4 > test.tsv
library(ggplot2)
library(factoextra)
library(tidyverse)
library(dplyr)
library(data.table)

# Read the data
file<- "test.tsv"
allele_freq_data <- fread(file, header = TRUE, sep = "\t")
pca_result<-readRDS("test_pca_result.rds")
pc_scores <- as.data.frame(pca_result$x[, 1:3])  # Assuming you want to plot PC1 vs PC2
populations<- (allele_freq_data$population)

# Combine PCA scores and population information into a data frame
pca_plot_data <- data.frame(
  PC1 = pc_scores[, 1],
  PC2 = pc_scores[, 2],
  PC3 = pc_scores[, 3],
  Population = populations
)

pca_plot_data$estuary <- sub("_.*", "", pca_plot_data$Population) #separate estuary from name
pca_plot_data$SampleDate <- sub("^[^_]*_", "", pca_plot_data$Population) #separate date from name

color_mapping <- c("Younger" = "#206450", #when making the map, changed to this colour because other too similar to light green
                   "Scott" = "#D95F02",
                   "Lombardi" = "#7570B3",
                   "OldDairy" = "#E7298A",
                   "Laguna" = "#66A61E",
                   "Waddell" = "#E6AB02")
pca_plot_data$color <- color_mapping[pca_plot_data$estuary] #defining colours by estuary

shape_mapping <- c("Spring.2016" ="o",  "Fall.2016" ="O")
pca_plot_data$shape <- shape_mapping[pca_plot_data$SampleDate] #defining shape by sampling time

pca_plot_data$shape <- as.factor(pca_plot_data$shape) #For some reason, this needs to be converted to cat!!
write.table(pca_plot_data, "pca-plot_data2016.tsv")

pca_colored_plot <- ggplot(pca_plot_data, aes(x = PC1, y = PC2, color = color, shape = factor(shape))) +
  geom_point(size = 3) +
  guides(color = guide_legend(title = "Estuary"), shape = guide_legend(title = "Sample Date")) +
  scale_color_manual(values = unique(pca_plot_data$color),
                     labels = setNames(unique(pca_plot_data$estuary), unique(pca_plot_data$color))) +
scale_shape_manual(values = c("o" = 1, "O" = 19),
                     labels = c("Spring-2016", "Fall-2016")) +
  labs(
    x = "Principal Component 1",
    y = "Principal Component 2",
    title = "PC 1 + 2 Allele Frequencies 2016") + theme_minimal()

ggsave("test_original.png", pca_colored_plot)



#Notes:
#When comparing my PCAs from popoolation 2 to poolfstat results (above), the PCAs are super different. 









print("data loaded")
pca_result <- prcomp(allele_freq_data[, -1], center = TRUE, scale = TRUE) #run pca on everything except text (population names in first column)
saveRDS(pca_result, file = paste0(sub(".tsv", "", file), "_pca_result.rds"))
write.table(pca_result$sdev, file = paste0(sub(".tsv", "", file), "_pca_result_sd.tsv"))
write.table(pca_result$rotation, file = paste0(sub(".tsv", "", file), "_pca_result_rotation.tsv"))
write.table(pca_result$x, file = paste0(sub(".tsv", "", file), "_pca_result_scores.tsv"))


## Now need to take results and run as an interactive job: ##
#adjusting what the files are 
cut -f1 test.tsv > test_populations.tsv


library(ggplot2)
library(factoextra)
library(tidyverse)
library(dplyr)
library(data.table)

pca_result<-readRDS("test_pca_result.rds")
file<- "test_populations.tsv"

allele_freq_data <- fread(file, header = TRUE, sep = "\t") 
#PReviously used the set keys stuff, but that lead to errors due to name reordering...
#setnames(allele_freq_data, old = names(allele_freq_data)[1], new = "ID")
#setkey(allele_freq_data, ID)
rownames(pca_result$x) <- allele_freq_data$population
#setattr(allele_freq_data, "row.names", allele_freq_data$ID)

# Process the data
allele_freq_data[, estuary := sapply(strsplit(population, "_"), `[`, 1)]
allele_freq_data[, season := sapply(strsplit(population, "_|-"), `[`, 2)]
allele_freq_data[, season := as.factor(season)]
allele_freq_data[, estuary := as.factor(estuary)]

#PCA biplot (plot that shows SNPs and populations with ID labels)
#doesn't run on interactive job, and times out as submitted job (with above section)
#snp_pop <- fviz_pca_biplot(pca_result, repel = TRUE, col.var = "#2E9FDF", col.ind = "#696969")
#ggsave(filename = paste0(sub(".tsv", "", file), "_pca_pop_snp.png"), plot = snp_pop, width = 8, height = 6, dpi = 300)

# Scree plot
scree_plot <- fviz_eig(pca_result)
ggsave(filename = paste0(sub(".tsv", "", file), "_scree_plot.png"), plot = scree_plot, width = 8, height = 6, dpi = 300)

# Populations only (coloured by estuary, shape by year)
pc_scores <- as.data.table(pca_result$x[, 1:3])
pc_scores[, Population := allele_freq_data$population]

pc_scores[, estuary := sub("_.*", "", Population)]
pc_scores[, SampleDate := sub("^[^_]*_", "", Population)]


shape_mapping <- c("Spring.2016" = "o", "Fall.2016" = "O", "Spring.2017" = "s", "Fall.2017" = "S")
pc_scores[, shape := shape_mapping[SampleDate]]
pc_scores[, shape := as.factor(shape)]

#Fix colour issue and add the Variance Explained on each PC:
explained_variance <- pca_result$sdev^2 / sum(pca_result$sdev^2) * 100
pc1_var <- round(explained_variance[1], 2)
pc2_var <- round(explained_variance[2], 2)


#Version 2:
pca_colored_plot <- ggplot(pc_scores, aes(x = PC1, y = PC2, color = estuary, shape = factor(shape))) +
  geom_point(size = 3, stroke = 1.2) +
  guides(color = guide_legend(title = "Estuary"), shape = guide_legend(title = "Sample Date")) +
  scale_color_manual(values = c("Younger" = "#2E2585", "Scott" = "#34B6FF", "Lombardi" = "#EADA02", "OldDairy" = "#D55E00", "Laguna" = "#287762", "Waddell" = "#A000E0")) +
  scale_shape_manual(values = c("o" = 1, "O" = 19, "s" = 0, "S" = 15), labels = c("Spring-2016", "Fall-2016", "Spring-2017", "Fall-2017")) +
  labs(x = paste0("PC1 (", pc1_var, "%)"), y = paste0("PC2 (", pc2_var, "%)")) + 
  theme_minimal() #Please note for 2017 datasets, need to rearrange the labels of scale_shape_manual (otherwise onyl 2016 labels printed...)

ggsave(filename = paste0(sub(".tsv", "", file), "_34testpca_plot.png"), plot = pca_colored_plot, width = 8, height = 6, dpi = 300)

